## tl;dr

A auditoria separa as 48 falhas estritas em 18 erros semânticos confirmados do agente, 27 casos não adjudicáveis sem corrigir pergunta/ground truth, 2 respostas adicionais aceitáveis e 1 falha operacional. Entre os 45 casos adjudicáveis, 27 são aceitos (60%). Esse número não substitui o benchmark; ele mostra por que o score estrito de 34,2% mistura qualidade do agente com qualidade da especificação.

## Context & Methods

### Key Assumptions

- O run auditado usa `gpt-5.4-nano`, `full_agent` e nenhum exemplo de ground truth.
- Um caso ambíguo não é contado como erro confirmado do agente.
- A classificação manual é reproduzida a partir de `failure_audit.json`; agregações são determinísticas.
- O hard-73 deve ser tratado como conjunto de desenvolvimento após esta auditoria.

## Data

Carrega os 73 registros do run e as 48 adjudicações salvas no diretório `audit/`.

In [ ]:
import json
from collections import Counter
from pathlib import Path

run_dir = Path.cwd()
if run_dir.name == 'audit':
    run_dir = run_dir.parent
audit = json.loads((run_dir / 'audit/failure_audit.json').read_text(encoding='utf-8'))
payload = json.loads((run_dir / 'variants/full_agent/results.json').read_text(encoding='utf-8'))
records = payload.get('records') or payload.get('results') or payload
cases = audit['cases']
assert len(records) == 73
assert len(cases) == 48

## Results

Reproduz a decomposição primária e verifica que ela reconcilia exatamente as 48 falhas.

In [ ]:
categories = Counter(case['audit_category'] for case in cases)
layers = Counter(case['primary_layer'] for case in cases)
strict_matches = sum(record.get('result_match') is True for record in records)
accepted_extra = categories['format_only'] + categories['acceptable_alternative_rejected']
indeterminate = categories['question_underspecified'] + categories['ground_truth_problem']
confirmed_errors = categories['agent_semantic_error']
adjudicable = strict_matches + accepted_extra + confirmed_errors
summary = {
    'strict_matches': strict_matches,
    'categories': dict(categories),
    'layers': dict(layers),
    'indeterminate': indeterminate,
    'accepted_rate_among_adjudicable': (strict_matches + accepted_extra) / adjudicable,
}
assert sum(categories.values()) == 48
summary

In [ ]:
by_id = {record['id']: record for record in records}
catalog_calls = Counter()
for case in cases:
    catalog_calls[case['audit_category']] += int(by_id[case['id']].get('catalog_tool_calls_count') or 0)
{
    'catalog_calls_total': sum(int(r.get('catalog_tool_calls_count') or 0) for r in records),
    'catalog_calls_by_category': dict(catalog_calls),
    'correction_cases': sum(bool(r.get('correction_attempts')) for r in records),
    'failure_records_with_warnings': sum(bool(by_id[c['id']].get('generated_sql_validation_warnings')) for c in cases),
}

## Takeaways

1. Corrigir o benchmark e congelar um novo teste vem antes de tuning.
2. Definições de UTI, data, idade e obstetrícia devem virar contratos de negócio independentes do teste.
3. Retrieval e validação precisam tratar cobertura, denominador, top-N por grupo e disponibilidade socioeconômica.
4. O comparador deve aceitar projeções extras, empates e representações equivalentes quando a semântica central coincide.
5. O banco de exemplos deve ser disjunto, versionado e criado sem observar o novo conjunto de teste.